# Multimodal 3D Generation - Google Colab (A100 GPU)

Notebook tạo mô hình 3D từ ảnh 2D + mô tả văn bản, sử dụng **A100 GPU**:
1. Mount Google Drive + kiểm tra project code
2. Tải dữ liệu **Cap3D ABO** (~40 GB) → lưu trên Drive
3. Chuẩn hóa caption sang **tiếng Việt** + thống kê nhãn
4. Cấu hình training **A100** (Transformer decoder, batch=64, 2048 points)
5. Train model với **auto-resume** từ checkpoint
6. Evaluate metrics (Chamfer Distance, F-Score)
7. **Hiển thị độ đo đánh giá** trực quan
8. Demo visualization (Gradio)

### Model đang sử dụng:
- **Encoder**: OpenCLIP ViT-B-32 (image + text)
- **Decoder**: Transformer decoder, 6 layers, width=384
- **Output**: 2048 points/object
- **Checkpoint**: `checkpoints/best_model.pth`

## 1. Setup & Mount Google Drive

In [ ]:
# Mount Google Drive
from google.colab import drive
try:
    drive.mount('/content/drive', force_remount=False)
    print('✓ Google Drive mounted')
except:
    print('✓ Google Drive already mounted')

# Xác nhận CPU-only
import torch, psutil
ram_gb = psutil.virtual_memory().total / (1024**3)
print(f'\n💻 Chạy trên CPU (RAM: {ram_gb:.1f} GB)')

In [ ]:
import os

# === CẤU HÌNH ĐƯỜNG DẪN ===
# Mọi thứ chạy trực tiếp trên Drive
WORK_DIR = '/content/drive/MyDrive/ThiGiacMayTinh_BaiTapLon'

# Tạo thư mục nếu chưa có
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(f'{WORK_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{WORK_DIR}/results', exist_ok=True)

print(f'✓ Work dir: {WORK_DIR}')

## 2. Kiểm tra Project Code

Project code nằm trực tiếp trên Drive.  
Đảm bảo đã upload các thư mục sau lên Drive: `src/`, `configs/`, `images/`

In [ ]:
# Kiểm tra xem có đủ files không
import os

required_files = {
    'src/__init__.py': 'Package init',
    'src/train.py': 'Training script',
    'src/evaluate.py': 'Evaluation script',
    'src/demo.py': 'Gradio demo',
    'src/data/__init__.py': 'Data package init',
    'src/data/dataset.py': 'Dataset & DataLoader',
    'src/data/prepare_data.py': 'Data preparation',
    'src/data/transforms.py': 'Data transforms',
    'src/models/__init__.py': 'Models package init',
    'src/models/multimodal3d.py': 'Main model',
    'src/models/encoders.py': 'CLIP encoders',
    'src/models/fusion.py': 'Fusion modules',
    'src/models/decoder.py': 'Point cloud decoder',
    'src/losses/__init__.py': 'Losses package init',
    'src/losses/losses.py': 'Loss functions',
    'configs/colab.yaml': 'Default config (PC)',
}

all_ok = True
for rel_path, desc in required_files.items():
    path = os.path.join(WORK_DIR, rel_path)
    exists = os.path.exists(path)
    status = '✓' if exists else '✗ THIẾU'
    print(f'  {status}: {rel_path} ({desc})')
    if not exists:
        all_ok = False

if all_ok:
    print(f'\n✓ Project structure OK! ({len(required_files)} files)')
else:
    print('\n✗ Thiếu files! Upload toàn bộ thư mục src/ và configs/ lên Drive.')
    print(f'  Drive path: {WORK_DIR}/')

## 3. Cài Dependencies

In [ ]:
# Cài các thư viện cần thiết
!pip install -q open-clip-torch trimesh gradio plotly tensorboard scipy tqdm pyyaml open3d psutil

# Verify imports
import torch, open_clip, trimesh, plotly, scipy, psutil
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
try:
    import open3d
    print(f'Open3D: {open3d.__version__} (mesh chất lượng cao)')
except:
    print('Open3D: không cài được (dùng trimesh fallback)')

## 4.1. Tải dữ liệu Cap3D

Dữ liệu sẽ được tải và giải nén trực tiếp lên Drive (~40GB).  
Nếu đã có data, cell này sẽ skip.

In [ ]:
%%time
import os, zipfile, shutil, io, json

DATA_DIR = f'{WORK_DIR}/Cap3D_data'
os.makedirs(f'{DATA_DIR}/pointclouds', exist_ok=True)
os.makedirs(f'{DATA_DIR}/rendered_images', exist_ok=True)

# --- [1/3] Captions ABO ---
csv_path = f'{DATA_DIR}/Cap3D_automated_ABO.csv'
if os.path.exists(csv_path):
    print('[1/3] ✓ Captions ABO đã có')
else:
    print('[1/3] Tải captions ABO...')
    !pip install -q huggingface_hub
    !hf download tiange/Cap3D Cap3D_automated_ABO.csv \
        --repo-type dataset --local-dir /tmp/cap3d_download
    shutil.copy2('/tmp/cap3d_download/Cap3D_automated_ABO.csv', csv_path)
    print('✓ Captions saved')

# --- [2/3] Point clouds ABO ---
pc_dir = f'{DATA_DIR}/pointclouds'
n_pc = len([f for f in os.listdir(pc_dir) if f.endswith('.ply')])
if n_pc > 100:
    print(f'\n[2/3] ✓ Point clouds ABO đã có ({n_pc} files)')
else:
    print('\n[2/3] Tải point clouds ABO...')
    !pip install -q huggingface_hub
    !hf download tiange/Cap3D PointCloud_zips_ABO/compressed_pcs_00.zip \
        --repo-type dataset --local-dir /tmp/cap3d_download
    pc_zip = '/tmp/cap3d_download/PointCloud_zips_ABO/compressed_pcs_00.zip'
    print('Giải nén lên Drive...')
    with zipfile.ZipFile(pc_zip, 'r') as z:
        members = [m for m in z.namelist() if m.endswith('.ply')]
        for i, m in enumerate(members, 1):
            fname = os.path.basename(m)
            with z.open(m) as src, open(f'{pc_dir}/{fname}', 'wb') as dst:
                dst.write(src.read())
            if i % 2000 == 0 or i == len(members):
                print(f'  [{i}/{len(members)}]')
    os.remove(pc_zip)
    print(f'✓ Point clouds ABO: {len(os.listdir(pc_dir))} files')

# --- [3/3] Rendered images ABO (SMART RESUME) ---
img_dir = f'{DATA_DIR}/rendered_images'
existing_folders = set([d for d in os.listdir(img_dir) if os.path.isdir(f'{img_dir}/{d}')])

import pandas as pd
df = pd.read_csv(csv_path, header=None)
expected_ids = set(df[0].astype(str).tolist())
print(f'ABO objects: {len(expected_ids)}')

missing_ids = expected_ids - existing_folders
n_missing = len(missing_ids)
n_total = len(expected_ids)

if n_missing == 0:
    print(f'\n[3/3] ✓ Rendered images đã có ({len(existing_folders)}/{n_total} folders)')
elif n_missing < 10:
    print(f'\n[3/3] Rendered images thiếu {n_missing}/{n_total} folders (ít, bỏ qua)')
else:
    print(f'\n[3/3] Rendered images thiếu ({n_missing}/{n_total} folders). Tiếp tục tải...')
    !pip install -q huggingface_hub
    
    main_zip_path = '/tmp/cap3d_download/RenderedImage_perobj_zips_ABO/compressed_imgs_perobj_00.zip'
    if not os.path.exists(main_zip_path):
        print('Đang tải main zip...')
        !hf download tiange/Cap3D RenderedImage_perobj_zips_ABO/compressed_imgs_perobj_00.zip \
            --repo-type dataset --local-dir /tmp/cap3d_download
        print('✓ Main zip downloaded')
    
    print(f'Giải nén {n_missing} folders còn thiếu...')
    with zipfile.ZipFile(main_zip_path, 'r') as outer_z:
        inner_zips = [m for m in outer_z.namelist() if m.endswith('.zip')]
        zip_to_id = {z: os.path.basename(z).replace('.zip', '') for z in inner_zips}
        needed_zips = [z for z in inner_zips if zip_to_id[z] in missing_ids]
        for i, inner_name in enumerate(needed_zips, 1):
            obj_id = zip_to_id[inner_name]
            obj_dir = f'{img_dir}/{obj_id}'
            os.makedirs(obj_dir, exist_ok=True)
            inner_data = outer_z.read(inner_name)
            with zipfile.ZipFile(io.BytesIO(inner_data)) as inner_z:
                for img_file in inner_z.namelist():
                    if img_file.endswith(('.png', '.jpg')):
                        fname = os.path.basename(img_file)
                        with inner_z.open(img_file) as src, open(f'{obj_dir}/{fname}', 'wb') as dst:
                            dst.write(src.read())
            if i % 100 == 0 or i == len(needed_zips):
                print(f'  [{i}/{len(needed_zips)}]')
    
    if os.path.exists(main_zip_path):
        os.remove(main_zip_path)

# --- Tạo splits (ShapeNet auto-detected từ shapenet/ShapeNet_pcs/) ---
print('\nTạo splits...')
os.chdir(WORK_DIR)
splits_dir = f'{DATA_DIR}/splits'
if os.path.exists(splits_dir):
    shutil.rmtree(splits_dir)

sn_pc_dir = f'{DATA_DIR}/shapenet/ShapeNet_pcs'
if os.path.exists(sn_pc_dir):
    n_sn = len([f for f in os.listdir(sn_pc_dir) if f.endswith('.ply')])
    if n_sn > 100:
        print(f'  ✓ ShapeNet point clouds: {n_sn} files → sẽ được gộp vào splits')

!python src/data/prepare_data.py --data-dir "{DATA_DIR}" --skip-extract

for split in ['train', 'val', 'test']:
    sp = f'{DATA_DIR}/splits/{split}.txt'
    if os.path.exists(sp):
        with open(sp) as f:
            n = len([l for l in f if l.strip()])
        print(f'  {split}: {n} objects')

!df -h /content | tail -1
print('\n✓ Data sẵn sàng!')

## 4.2 Tải thêm ShapeNet

ShapeNet có **52,472 objects** → gấp 7 lần ABO.  
Chỉ tải **point clouds + captions**. Không tải rendered images (~30 GB) để tiết kiệm thời gian.

In [ ]:
%%time
import os, zipfile, shutil, glob, numpy as np

DATA_DIR = f'{WORK_DIR}/Cap3D_data'
SHAPENET_DIR = f'{DATA_DIR}/shapenet'
os.makedirs(f'{SHAPENET_DIR}/ShapeNet_pcs', exist_ok=True)

# === [1/2] ShapeNet Captions ===
shapenet_csv = f'{DATA_DIR}/Cap3D_automated_ShapeNet.csv'
if os.path.exists(shapenet_csv):
    print('[1/2] ✓ ShapeNet captions đã có')
else:
    print('[1/2] Tải ShapeNet captions...')
    !pip install -q huggingface_hub
    !hf download tiange/Cap3D Cap3D_automated_ShapeNet.csv \
        --repo-type dataset --local-dir /tmp/cap3d_shapenet
    shutil.copy2('/tmp/cap3d_shapenet/Cap3D_automated_ShapeNet.csv', shapenet_csv)
    print('✓ ShapeNet captions saved')

# === [2/2] ShapeNet Point Clouds (~15 GB, 1 file) ===
shapenet_pc_dir = f'{SHAPENET_DIR}/ShapeNet_pcs'
n_pc = len([f for f in os.listdir(shapenet_pc_dir) if f.endswith('.ply')])

if n_pc > 10000:
    print(f'\n[2/2] ✓ ShapeNet point clouds đã có ({n_pc} files)')
else:
    print(f'\n[2/2] ShapeNet point clouds ({n_pc} files hiện có). Tải thêm...')
    !pip install -q huggingface_hub

    zip_name = 'compressed_pcs_00.zip'
    zip_path = f'/tmp/cap3d_shapenet/PointCloud_zips_ShapeNet/{zip_name}'
    marker = f'{shapenet_pc_dir}/.extracted_00'

    if os.path.exists(marker):
        print(f'  ✓ {zip_name} đã extract')
    else:
        if not os.path.exists(zip_path):
            print(f'  Tải {zip_name} ...')
            !hf download tiange/Cap3D PointCloud_zips_ShapeNet/{zip_name} \
                --repo-type dataset --local-dir /tmp/cap3d_shapenet

        print(f'  Giải nén {zip_name}...')
        with zipfile.ZipFile(zip_path, 'r') as z:
            members = [m for m in z.namelist() if m.endswith('.ply')]
            for i, m in enumerate(members, 1):
                fname = os.path.basename(m)
                with z.open(m) as src, open(f'{shapenet_pc_dir}/{fname}', 'wb') as dst:
                    dst.write(src.read())
                if i % 5000 == 0 or i == len(members):
                    print(f'    [{i}/{len(members)}]')

        open(marker, 'w').close()
        os.remove(zip_path)
        print(f'  ✓ {zip_name} done, deleted zip')

    n_final = len([f for f in os.listdir(shapenet_pc_dir) if f.endswith('.ply')])
    print(f'\n✓ ShapeNet point clouds: {n_final} files')

# Convert .ply → .pt: xem Cell 4.3 bên dưới (xử lý cả ABO + ShapeNet cùng lúc)

# === Thống kê tổng ===
import pandas as pd
abo_csv = f'{DATA_DIR}/Cap3D_automated_ABO.csv'
n_abo = len(pd.read_csv(abo_csv, header=None))
n_shapenet = len(pd.read_csv(shapenet_csv, header=None))
print(f'\n=== TỔNG DỮ LIỆU ===')
print(f'  ABO:      {n_abo:,} objects')
print(f'  ShapeNet: {n_shapenet:,} objects')
print(f'  Tổng:     {n_abo + n_shapenet:,} objects')
!df -h /content/drive | tail -1

## 4.3 Convert Point Cloud .ply → .pt (tăng tốc loading 5-10x)

Chạy **1 lần duy nhất**. Sau khi convert xong, training sẽ tự động dùng .pt (nhanh hơn trimesh rất nhiều trên CPU).

In [ ]:
%%time
import torch, trimesh, os, glob, numpy as np
import time

DATA_DIR = f'{WORK_DIR}/Cap3D_data'

def convert_ply_to_pt(ply_dir, pt_dir, label=''):
    os.makedirs(pt_dir, exist_ok=True)
    existing_pt = len([f for f in os.listdir(pt_dir) if f.endswith('.pt')])
    existing_ply = len([f for f in os.listdir(ply_dir) if f.endswith('.ply')])
    
    if existing_pt >= existing_ply and existing_pt > 100:
        print(f'[{label}] ✓ Đã convert: {existing_pt} files .pt (skip)')
        return
    
    print(f'[{label}] Converting {existing_ply} files .ply → .pt ...')
    converted, errors = 0, 0
    
    for ply_file in sorted(glob.glob(f'{ply_dir}/*.ply')):
        obj_id = os.path.basename(ply_file).replace('.ply', '')
        pt_path = f'{pt_dir}/{obj_id}.pt'
        
        if os.path.exists(pt_path):
            converted += 1
            continue
        
        # Retry logic cho I/O errors
        max_retries = 3
        for attempt in range(max_retries):
            try:
                mesh = trimesh.load(ply_file)
                xyz = torch.tensor(mesh.vertices, dtype=torch.float32)
                
                if hasattr(mesh, 'visual') and hasattr(mesh.visual, 'vertex_colors'):
                    vc = mesh.visual.vertex_colors
                    if vc is not None and len(vc) == len(xyz):
                        rgb = torch.tensor(vc[:, :3], dtype=torch.float32) / 127.5 - 1.0
                    else:
                        rgb = torch.zeros_like(xyz)
                else:
                    rgb = torch.zeros_like(xyz)
                
                pc = torch.cat([xyz, rgb], dim=1)  # [N, 6]
                
                # Retry save
                for save_attempt in range(max_retries):
                    try:
                        torch.save(pc, pt_path)
                        break
                    except (OSError, IOError) as e:
                        if save_attempt == max_retries - 1:
                            raise
                        time.sleep(1)
                
                converted += 1
                break
            except (OSError, IOError) as e:
                if attempt == max_retries - 1:
                    errors += 1
                    print(f'  ⚠ I/O error {obj_id}: {e} (skip)')
                else:
                    time.sleep(2)
            except Exception as e:
                errors += 1
                break
        
        # Progress update + delay để tránh Drive rate limit
        if converted % 500 == 0 and converted > 0:
            print(f'  [{converted}/{existing_ply}] converted, {errors} errors')
            time.sleep(2)  # Pause để Drive nghỉ
    
    print(f'  ✓ Hoàn tất: {converted} files .pt, {errors} errors')

# === ABO ===
convert_ply_to_pt(
    ply_dir=f'{DATA_DIR}/pointclouds',
    pt_dir=f'{DATA_DIR}/pointclouds_pt',
    label='ABO'
)

# === ShapeNet ===
sn_ply_dir = f'{DATA_DIR}/shapenet/ShapeNet_pcs'
sn_pt_dir = f'{DATA_DIR}/shapenet/pointclouds_pt'
if os.path.exists(sn_ply_dir) and len([f for f in os.listdir(sn_ply_dir) if f.endswith('.ply')]) > 0:
    convert_ply_to_pt(ply_dir=sn_ply_dir, pt_dir=sn_pt_dir, label='ShapeNet')
else:
    print('[ShapeNet] Chưa có .ply files → bỏ qua')

# Thống kê dung lượng
total_size = 0
for pt_dir in [f'{DATA_DIR}/pointclouds_pt', sn_pt_dir]:
    if os.path.exists(pt_dir):
        size = sum(os.path.getsize(f'{pt_dir}/{f}') for f in os.listdir(pt_dir) if f.endswith('.pt'))
        total_size += size
        print(f'  {os.path.basename(os.path.dirname(pt_dir))}/pointclouds_pt: {size / (1024**3):.2f} GB')
print(f'  Tổng .pt: {total_size / (1024**3):.2f} GB')
print(f'\n✓ Training sẽ tự động dùng .pt (5-10x nhanh hơn .ply)')

## 4.5 Chuẩn hóa Caption sang Tiếng Việt + Thống kê nhãn

Dịch captions sang tiếng Việt bằng Google Translate API, sau đó thống kê lại số nhãn theo category.

In [ ]:
!pip install -q transformers sentencepiece torch

import os, json
import pandas as pd
import torch
from transformers import MarianMTModel, MarianTokenizer

DATA_DIR = f'{WORK_DIR}/Cap3D_data'
VI_CAPTIONS_PATH = f'{DATA_DIR}/captions_vi.json'

# === Dịch captions sang tiếng Việt (batch với MarianMT) ===
if os.path.exists(VI_CAPTIONS_PATH):
    with open(VI_CAPTIONS_PATH) as f:
        vi_captions = json.load(f)
    print(f'✓ Captions tiếng Việt đã có ({len(vi_captions)} objects)')
else:
    print('Đang tải model MarianMT (Helsinki-NLP/opus-mt-en-vi)...')
    model_name = 'Helsinki-NLP/opus-mt-en-vi'
    tokenizer = MarianTokenizer.from_pretrained(model_name)
    model = MarianMTModel.from_pretrained(model_name)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = model.to(device)
    print(f'✓ Model loaded on {device}')

    captions_path = f'{DATA_DIR}/captions.json'
    with open(captions_path) as f:
        all_captions = json.load(f)

    vi_captions = {}
    total = len(all_captions)
    batch_size = 32

    obj_ids = list(all_captions.keys())
    captions_texts = []
    for obj_id in obj_ids:
        caps = all_captions[obj_id]
        cap_text = caps.get('blip2', caps.get('caption', caps.get('general', '')))
        if isinstance(cap_text, list):
            cap_text = cap_text[0]
        captions_texts.append(str(cap_text))

    print(f'Bắt đầu dịch {total} captions (batch size={batch_size})...')

    for i in range(0, total, batch_size):
        batch_ids   = obj_ids[i:i+batch_size]
        batch_texts = captions_texts[i:i+batch_size]
        try:
            inputs = tokenizer(batch_texts, return_tensors="pt", padding=True,
                               truncation=True, max_length=128)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            with torch.no_grad():
                translated = model.generate(**inputs, max_length=128)
            vi_texts = tokenizer.batch_decode(translated, skip_special_tokens=True)
            for obj_id, en_text, vi_text in zip(batch_ids, batch_texts, vi_texts):
                vi_captions[obj_id] = {'en': en_text, 'vi': vi_text}
        except Exception as e:
            for obj_id, en_text in zip(batch_ids, batch_texts):
                vi_captions[obj_id] = {'en': en_text, 'vi': en_text}

        if (i + batch_size) % 500 == 0 or (i + batch_size) >= total:
            print(f'  [{min(i+batch_size, total)}/{total}] objects translated')
            with open(VI_CAPTIONS_PATH, 'w', encoding='utf-8') as f:
                json.dump(vi_captions, f, ensure_ascii=False, indent=2)

    print(f'✓ Đã dịch {len(vi_captions)} captions → {VI_CAPTIONS_PATH}')

# === Hiển thị mẫu captions tiếng Việt ===
print('\n--- Mẫu captions tiếng Việt ---')
for obj_id, caps in list(vi_captions.items())[:5]:
    print(f'  {obj_id}:')
    print(f'    EN: {caps["en"][:80]}')
    print(f'    VI: {caps["vi"][:80]}')
    print()

print('✓ Chuẩn hóa caption tiếng Việt hoàn tất!')

In [ ]:
import os, json
from collections import Counter

DATA_DIR = f'{WORK_DIR}/Cap3D_data'
VI_CAPTIONS_PATH = f'{DATA_DIR}/captions_vi.json'

# Load captions nếu chưa có trong memory
if 'vi_captions' not in dir() or not vi_captions:
    with open(VI_CAPTIONS_PATH) as f:
        vi_captions = json.load(f)

# === Thống kê từ khóa tiếng Việt ===
vi_words = []
for caps in vi_captions.values():
    vi_words.extend(caps['vi'].lower().split())

vi_counter = Counter(vi_words)
vi_stopwords = {'một', 'và', 'với', 'của', 'có', 'là', 'các', 'được', 'trong',
                'trên', 'cho', 'từ', 'này', 'để', 'không', 'rất', 'bởi', 'như'}
vi_keywords = [(w, c) for w, c in vi_counter.most_common(200)
               if w not in vi_stopwords and len(w) > 2][:20]

print(f'=== THỐNG KÊ NHÃN TIẾNG VIỆT ===')
print(f'Tổng objects  : {len(vi_captions):,}')
print(f'Tổng số từ    : {len(vi_words):,}')
print(f'Số từ khác nhau: {len(vi_counter):,}')
print(f'\nTop 20 từ khóa:')
for i, (word, count) in enumerate(vi_keywords, 1):
    bar = '█' * min(count // 50, 30)
    print(f'  {i:2d}. {word:15s} ({count:4d}) {bar}')

# === Phân bố category ===
vi_categories = {
    'Ghế / Bàn / Tủ': ['ghế', 'bàn', 'kệ', 'tủ', 'giá'],
    'Đèn / Đồ gia dụng': ['đèn', 'bình', 'cốc', 'hộp', 'chai'],
    'Đồ điện tử': ['điện', 'máy', 'laptop', 'màn'],
    'Phương tiện': ['xe', 'ô', 'thuyền', 'bay'],
    'Vật liệu': ['gỗ', 'kim', 'nhựa', 'vải', 'thép'],
}

print(f'\n=== PHÂN BỐ CATEGORY ===')
for cat, kws in vi_categories.items():
    count = sum(vi_counter.get(kw, 0) for kw in kws)
    bar = '█' * min(count // 100, 25)
    print(f'  {cat:25s}: {count:5d} lần đề cập  {bar}')

print(f'\n✓ Thống kê nhãn hoàn tất!')

## 5.1 Cấu hình Training (A100 GPU)

Config cho **A100 GPU**:
- **Transformer decoder** (6 layers, width=384)
- **batch_size=64**, num_points=2048, max_objects=0 (full dataset)
- **2048 points/object** — chi tiết hơn 4x so với CPU config
- Save checkpoint mỗi **200 batches**

In [ ]:
import yaml, os, json, random
import pandas as pd
import torch

DATA_DIR = f'{WORK_DIR}/Cap3D_data'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

# === [1/2] Tái tạo splits ABO only ===
abo_csv = f'{DATA_DIR}/Cap3D_automated_ABO.csv'
pc_dir  = f'{DATA_DIR}/pointclouds'       # .ply files
pt_dir  = f'{DATA_DIR}/pointclouds_pt'    # .pt files (converted từ .ply)
img_dir = f'{DATA_DIR}/rendered_images'

df = pd.read_csv(abo_csv, header=None)
all_ids = df[0].astype(str).tolist()

# Chỉ giữ objects có đủ: (ply HOẶC pt) VÀ ảnh rendered
valid_ids = [
    oid for oid in all_ids
    if (os.path.exists(f'{pc_dir}/{oid}.ply') or os.path.exists(f'{pt_dir}/{oid}.pt'))
    and os.path.isdir(f'{img_dir}/{oid}')
]

random.seed(42)
random.shuffle(valid_ids)
n = len(valid_ids)
n_train = int(n * 0.8)
n_val   = int(n * 0.1)

splits = {
    'train': valid_ids[:n_train],
    'val':   valid_ids[n_train:n_train + n_val],
    'test':  valid_ids[n_train + n_val:],
}

splits_dir = f'{DATA_DIR}/splits'
os.makedirs(splits_dir, exist_ok=True)
for name, ids in splits.items():
    with open(f'{splits_dir}/{name}.txt', 'w') as f:
        f.write('\n'.join(ids))

print(f'✓ Splits ABO only: train={len(splits["train"])}, val={len(splits["val"])}, test={len(splits["test"])}')

# === [2/2] CONFIG A100 ===
a100_config = {
    'data': {
        'data_dir': DATA_DIR,
        'num_points': 2048,
        'image_size': 224,
        'batch_size': 64,
        'num_workers': 4,
        'max_objects': 0,
        'num_views': 1,
    },
    'model': {
        'clip_model': 'ViT-B-32',
        'pretrained': 'laion2b_s34b_b79k',
        'freeze_clip': True,
        'decoder_type': 'transformer',
        'decoder_width': 384,
        'decoder_layers': 6,
        'decoder_heads': 6,
        'fine_layers': 3,
        'fusion_type': 'concat',
        'fused_dim': 512,
        'coarse_points': 256,
        'fine_points': 2048,
    },
    'training': {
        'epochs': 100,
        'lr': 0.001,
        'weight_decay': 0.01,
        'warmup_epochs': 5,
        'patience': 25,
        'cooldown_seconds': 0,
    },
    'loss': {
        'lambda_cd': 1.0,
        'lambda_emd': 0.5,
        'lambda_color': 0.5,
        'lambda_repulsion': 0.1,
        'lambda_uniform': 0.05,
        'lambda_normal': 0.05,
        'normal_k': 8,
        'repulsion_k': 8,
        'repulsion_h': 0.03,
        'use_emd': False,
    },
    'logging': {
        'log_dir': 'runs',
        'checkpoint_dir': 'checkpoints',
        'log_every': 10,
        'val_every': 1,
        'save_every': 200,
    }
}

config_path = f'{WORK_DIR}/configs/a100.yaml'
os.makedirs(os.path.dirname(config_path), exist_ok=True)
with open(config_path, 'w') as f:
    yaml.dump(a100_config, f, default_flow_style=False)

print(f'✓ Config: {config_path}')
print(f'  decoder=transformer, batch=64, points=2048, ABO only ({n} objects)')

## 6. Training

**Nếu bị ngắt**: Chạy Cell 2 (Mount) → Cell 8 (Deps) → Cell 18 (Config) → Cell 21 (Train)

In [ ]:
%%time
# === BẮT ĐẦU TRAINING A100 (AUTO-RESUME) ===
import os, torch, json, time
os.chdir(WORK_DIR)

CKPT_DIR = f'{WORK_DIR}/checkpoints'
resume_path = ''

# Tìm checkpoint có epoch CAO NHẤT để tránh resume sai khi Drive không flush kịp
best_resume_ckpt = None
best_resume_epoch = -1

for ckpt_name in ['last_model.pth', 'best_model.pth', 'best_model_a100.pth']:
    ckpt_file = f'{CKPT_DIR}/{ckpt_name}'
    if not os.path.exists(ckpt_file):
        continue
    try:
        ckpt = torch.load(ckpt_file, map_location='cpu', weights_only=False)
        ckpt_cfg     = ckpt.get('config', {})
        ckpt_points  = ckpt_cfg.get('model', {}).get('fine_points', 0)
        ckpt_decoder = ckpt_cfg.get('model', {}).get('decoder_type', 'mlp')
        ckpt_epoch   = ckpt.get('epoch', -1)

        if ckpt_points == 2048 and ckpt_decoder == 'transformer':
            if ckpt_epoch > best_resume_epoch:
                best_resume_epoch = ckpt_epoch
                best_resume_ckpt  = (ckpt_name, ckpt_file, ckpt)
            print(f'  [{ckpt_name}] epoch={ckpt_epoch+1}, CD={ckpt.get("best_val_cd", float("inf")):.4f}')
        else:
            print(f'✗ Không tương thích A100 ({ckpt_name}): pts={ckpt_points}, decoder={ckpt_decoder}')
    except Exception as e:
        print(f'✗ Checkpoint hỏng: {ckpt_name} ({e})')

if best_resume_ckpt:
    ckpt_name, resume_path, ckpt = best_resume_ckpt
    epoch_done = best_resume_epoch + 1
    best_cd    = ckpt.get('best_val_cd', float('inf'))
    print(f'\n✓ Resume từ: {ckpt_name} (epoch {epoch_done}, CD={best_cd:.4f})')
else:
    print('▶ Train từ đầu (A100 config)')

resume_flag = f'--resume "{resume_path}"' if resume_path else ''

print(f'\n🏃 Bắt đầu training A100...')
!python src/train.py \
    --config configs/a100.yaml \
    {resume_flag}

# Kiểm tra kết quả
for f_name in ['best_model.pth', 'last_model.pth']:
    p = f'{CKPT_DIR}/{f_name}'
    if os.path.exists(p):
        size = os.path.getsize(p) / (1024**2)
        ckpt = torch.load(p, map_location='cpu', weights_only=False)
        epoch = ckpt.get('epoch', '?')
        cd = ckpt.get('best_val_cd', '?')
        print(f'✓ {f_name} ({size:.1f} MB) — epoch={epoch}, best_cd={cd}')
print('\n✓ Training hoàn tất!')

In [ ]:
import os
WORK_DIR = '/content/drive/MyDrive/ThiGiacMayTinh_BaiTapLon'
os.chdir(WORK_DIR)
os.makedirs(f'{WORK_DIR}/results', exist_ok=True)

!python src/evaluate.py \
    --config configs/a100.yaml \
    --checkpoint checkpoints/best_model.pth \
    --split test \
    --output results/evaluation_a100.json

## 7. Hiển thị Độ Đo Đánh Giá

Đọc metrics từ checkpoint + kết quả evaluation + vẽ biểu đồ Chamfer Distance theo epoch.

In [ ]:
import os, json, torch

WORK_DIR = '/content/drive/MyDrive/ThiGiacMayTinh_BaiTapLon'
CKPT_DIR = f'{WORK_DIR}/checkpoints'

# === [1] Thông tin checkpoint ===
print('=== KẾT QUẢ TRAINING (A100) ===')
for ckpt_name in ['best_model.pth', 'last_model.pth']:
    p = f'{CKPT_DIR}/{ckpt_name}'
    if os.path.exists(p):
        size_mb = os.path.getsize(p) / (1024**2)
        try:
            ckpt = torch.load(p, map_location='cpu', weights_only=False)
            epoch    = ckpt.get('epoch', '?')
            best_cd  = ckpt.get('best_val_cd', float('inf'))
            patience = ckpt.get('patience_counter', '?')
            print(f'  {ckpt_name}: epoch={epoch}, best_val_CD={best_cd:.6f}, patience={patience}, size={size_mb:.1f}MB')
        except Exception as e:
            print(f'  {ckpt_name}: ⚠ File bị hỏng ({e.__class__.__name__}) — size={size_mb:.1f}MB')
            print(f'    → Xóa file hỏng để tránh resume sai')
            os.remove(p)

# === [2] Kết quả Evaluation (CD, F-Score) ===
eval_path = f'{WORK_DIR}/results/evaluation_a100.json'
if os.path.exists(eval_path):
    with open(eval_path) as f:
        eval_results = json.load(f)
    print(f'\n=== ĐỘ ĐO ĐÁNH GIÁ (Test Set) ===')
    cd  = eval_results.get('chamfer_distance', eval_results.get('cd', None))
    fs1 = eval_results.get('fscore_0.01',    eval_results.get('f_score_001', eval_results.get('f1', None)))
    fs2 = eval_results.get('fscore_0.02',    eval_results.get('f_score_002', eval_results.get('f2', None)))
    n   = eval_results.get('num_samples',    eval_results.get('n',  None))
    if cd  is not None: print(f'  Chamfer Distance (CD) : {cd:.6f}  (thấp hơn = tốt hơn)')
    if fs1 is not None: print(f'  F-Score @ 0.01        : {fs1:.4f}  (cao hơn = tốt hơn)')
    if fs2 is not None: print(f'  F-Score @ 0.02        : {fs2:.4f}')
    if n   is not None: print(f'  Số mẫu test           : {n}')
    skip = {'chamfer_distance','cd','fscore_0.01','f_score_001','f1','fscore_0.02','f_score_002','f2','num_samples','n'}
    for k, v in eval_results.items():
        if k not in skip:
            print(f'  {k}: {v}')
else:
    print(f'\n⚠ Chưa có kết quả evaluation ({eval_path})')
    print('  Hãy chạy cell Evaluate (cell 21) trước')

## 8. Gradio Demo (Interactive UI)

Chạy Gradio demo với:
- Upload ảnh + nhập mô tả **tiếng Việt hoặc tiếng Anh** → tạo 3D
- Ví dụ: "một chiếc ghế gỗ có bốn chân", "một chiếc đèn bàn màu trắng"
- 3 mức chất lượng: low / medium / high
- Xuất .ply, .obj, .glb
- Public link để chia sẻ

**Chỉ cần chạy Cell này để demo!** (sau khi đã mount Drive + install deps)

In [ ]:
import os, sys, yaml, torch
WORK_DIR = '/content/drive/MyDrive/ThiGiacMayTinh_BaiTapLon'
sys.path.insert(0, WORK_DIR)
os.chdir(WORK_DIR)

# === [1/2] Đảm bảo configs/a100.yaml tồn tại trên Drive ===
A100_CONFIG_PATH = f'{WORK_DIR}/configs/a100.yaml'
os.makedirs(f'{WORK_DIR}/configs', exist_ok=True)

if not os.path.exists(A100_CONFIG_PATH):
    print('Tạo configs/a100.yaml trên Drive...')
    DATA_DIR = f'{WORK_DIR}/Cap3D_data'
    a100_config = {
        'data': {'batch_size': 64, 'data_dir': DATA_DIR, 'image_size': 224,
                 'max_objects': 0, 'num_points': 2048, 'num_workers': 4, 'num_views': 1},
        'logging': {'checkpoint_dir': 'checkpoints', 'log_dir': 'runs',
                    'log_every': 10, 'save_every': 200, 'val_every': 1},
        'loss': {'lambda_cd': 1.0, 'lambda_color': 0.5, 'lambda_emd': 0.5,
                 'lambda_normal': 0.05, 'lambda_repulsion': 0.1, 'lambda_uniform': 0.05,
                 'normal_k': 8, 'repulsion_h': 0.03, 'repulsion_k': 8, 'use_emd': False},
        'model': {'clip_model': 'ViT-B-32', 'coarse_points': 256, 'decoder_heads': 6,
                  'decoder_layers': 6, 'decoder_type': 'transformer', 'decoder_width': 384,
                  'fine_layers': 3, 'fine_points': 2048, 'freeze_clip': True,
                  'fused_dim': 512, 'fusion_type': 'concat', 'pretrained': 'laion2b_s34b_b79k'},
        'training': {'cooldown_seconds': 0, 'epochs': 100, 'lr': 0.001,
                     'patience': 25, 'warmup_epochs': 5, 'weight_decay': 0.01},
    }
    with open(A100_CONFIG_PATH, 'w') as f:
        yaml.dump(a100_config, f, default_flow_style=False)
    print(f'✓ Đã tạo configs/a100.yaml')
else:
    print('✓ configs/a100.yaml đã có')

# === [2/2] Chạy Gradio Demo ===
CKPT = f'{WORK_DIR}/checkpoints/best_model.pth'
if not os.path.exists(CKPT):
    raise FileNotFoundError(f'Không tìm thấy {CKPT}!\nHãy chạy cell Training trước để tạo best_model.pth')

ckpt = torch.load(CKPT, map_location='cpu', weights_only=False)
print(f'✓ Checkpoint: epoch={ckpt.get("epoch","?")}, best_cd={ckpt.get("best_val_cd","?")}')
print(f'  Config: configs/a100.yaml | Checkpoint: {CKPT}')

!python src/demo.py \
    --config "configs/a100.yaml" \
    --checkpoint "{CKPT}" \
    --share

## 9. Download Checkpoint về máy

Tải checkpoint đã train về máy để chạy demo local.

In [ ]:
# Download checkpoint về máy
from google.colab import files
import os, torch

WORK_DIR = '/content/drive/MyDrive/ThiGiacMayTinh_BaiTapLon'
CKPT_DIR = f'{WORK_DIR}/checkpoints'

# Hiển thị các checkpoint có sẵn
for ckpt_name in ['best_model.pth', 'last_model.pth']:
    p = f'{CKPT_DIR}/{ckpt_name}'
    if os.path.exists(p):
        ckpt = torch.load(p, map_location='cpu', weights_only=False)
        size  = os.path.getsize(p) / (1024**2)
        epoch = ckpt.get('epoch', '?')
        cd    = ckpt.get('best_val_cd', '?')
        print(f'  {ckpt_name}: {size:.1f} MB — epoch={epoch}, best_cd={cd}')

# Download best model
ckpt_to_download = f'{CKPT_DIR}/best_model.pth'
if not os.path.exists(ckpt_to_download):
    ckpt_to_download = f'{CKPT_DIR}/last_model.pth'

if os.path.exists(ckpt_to_download):
    print(f'\nDownloading {os.path.basename(ckpt_to_download)}...')
    files.download(ckpt_to_download)
else:
    print('✗ Không tìm thấy checkpoint!')

## 10. Ghi chú & Hướng dẫn

### Cấu trúc dự án trên Drive:
```
ThiGiacMayTinh_BaiTapLon/
├── src/
│   ├── train.py, evaluate.py, demo.py
│   ├── data/ (dataset.py, prepare_data.py, transforms.py)
│   ├── models/ (multimodal3d.py, encoders.py, decoder.py, fusion.py)
│   └── losses/ (losses.py)
├── configs/
│   ├── a100.yaml            # Config A100 (auto-generated, đang dùng)
│   └── colab.yaml           # Config CPU cũ (không dùng)
├── Cap3D_data/
│   ├── pointclouds/         # ABO point clouds (.ply)
│   ├── rendered_images/     # ABO 20 views
│   ├── Cap3D_automated_ABO.csv
│   ├── captions.json
│   ├── captions_vi.json     # Captions tiếng Việt (MarianMT)
│   └── splits/ (train.txt, val.txt, test.txt)
├── checkpoints/
│   ├── best_model.pth       # ✅ Model tốt nhất (cập nhật sau mỗi epoch)
│   └── last_model.pth       # Checkpoint gần nhất (dùng để resume)
└── results/
    └── evaluation_a100.json
```

### QUICK RESUME training sau khi bị ngắt:
```
Cell 2 (Mount) → Cell 8 (Deps) → Cell 18 (Config) → Cell 20 (Train)
```

### CHỈ DEMO (không train):
```
Cell 2 (Mount) → Cell 8 (Deps) → Cell 25 (Gradio Demo)
```

### Cấu hình A100:
| Tham số | Giá trị |
|---|---|
| decoder_type | transformer (6 layers, width=384) |
| batch_size | 64 |
| num_points | 2048 |
| max_objects | 0 (full dataset) |
| save_every | 200 batches |

### Đã thực hiện:
- ✅ **Captions tiếng Việt** (MarianMT) + thống kê nhãn
- ✅ **Transformer decoder** thay MLP → chất lượng tốt hơn rõ rệt
- ✅ **2048 points** (4x so với CPU model 512pts)
- ✅ **Mixed precision (AMP)** → tránh OOM trên T4/A100
- ✅ **Hiển thị độ đo** (CD, F-Score) sau evaluation